In [20]:
from dotenv import load_dotenv
import os
import faiss
from langchain_community.vectorstores import FAISS
from langchain_google_genai import(
    GoogleGenerativeAIEmbeddings,
    ChatGoogleGenerativeAI
)
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser

In [21]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings_hf = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [22]:
#load environment variables 
load_dotenv()
os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001",output_dimension=786)
dimension = len(embeddings_hf.embed_query("test"))
print(f"Length of embedding vector: {dimension}")

Length of embedding vector: 384


In [23]:
#file path of the document to be loaded
file_path=r"D:\GEN AI BASICS\RAG\Vector-database\Data\llama2-research-paper.pdf"
loader = PyPDFLoader(file_path)
pages=loader.load()
print("File loaded successfully")
print(f"number of pages in the document: {len(pages)}")

File loaded successfully
number of pages in the document: 77


In [24]:
#creating a chunker to split the document into smaller chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter=RecursiveCharacterTextSplitter(chunk_size=2000, 
chunk_overlap=200,
separators=["\n\n","\n"," ",""])
chunker=text_splitter.split_documents(pages)
print(f"number of chunks created: {len(chunker)}")

number of chunks created: 175


# LangChain FAISS VectorStore Architecture

LangChain FAISS VectorStore<br>
│<br>
├── FAISS Index<br>
│&nbsp;&nbsp;&nbsp;└── Stores numerical embedding vectors<br>
│<br>
├── InMemoryDocstore<br>
│&nbsp;&nbsp;&nbsp;└── Stores LangChain Document objects<br>
│&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;├── page_content<br>
│&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;└── metadata<br>
│<br>
└── index_to_docstore_id<br>
&nbsp;&nbsp;&nbsp;&nbsp;└── Maps FAISS vector position to the corresponding Document ID

In [9]:
# Creating a HNSW index
M = 32

faiss_index = faiss.IndexHNSWFlat(
    dimension,
    M,
    faiss.METRIC_L2
)

faiss_index.hnsw.efConstruction = 200
faiss_index.hnsw.efSearch = 64

In [10]:
#create a vector store using FAISS
vectorstore = FAISS(
    embedding_function=embeddings_hf,
    index=faiss_index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
    normalize_L2=True
)

In [11]:
#Add chunks to the vector store
documentsids=vectorstore.add_documents(chunker)
print(f"Documents added: {len(documentsids)}")
print(f"total vector: {vectorstore.index.ntotal}")

Documents added: 175
total vector: 175


In [12]:
#create a retriver to retriever
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})
print("Retriever created successfully")

Retriever created successfully


In [ ]:
#create a local rag prompt
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a question-answering assistant.

Answer the question only from the supplied context.

If the answer is not present in the context, say:
"I don't have enough information in the provided document."

Context:
{context}

Question:
{question}

Answer:
""")
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

In [18]:
#creating the llm
model=ChatGoogleGenerativeAI(model="gemini-2.5-flash",
temperature=0)
#creating the rag chain
rag_chain=(
    {
        "context": retriever|format_docs,
        "question": RunnablePassthrough(),


    }
    |prompt
    |model
    |StrOutputParser()
)

In [19]:
while True:
    user_input = input("Enter your question: ")
    if user_input.lower() in ["exit", "quit"]:
        print("Exiting the program.")
        break
    output = rag_chain.invoke(user_input)   
    print(f"Answer: {output}")

Answer: Llama 2 is an auto-regressive language model that uses an optimized transformer architecture. The tuned versions use supervised fine-tuning (SFT) and reinforcement learning with human feedback (RLHF) to align to human preferences for helpfulness and safety.
Exiting the program.


In [25]:
vectorstore.save_local("faiss_index_llama2")